# Brain Tumor Classification Using Feature Selection
This notebook demonstrates a complete pipeline for brain tumor classification using clinical and methylation data, with feature selection using Random Forest.

## Essential Imports

In [2]:
# Data Handling
import pandas as pd
import numpy as np
import os

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Explainability
import shap
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")
random_state = 42

## 📂 Load Data (Clinical & Methylation)

In [3]:
# Set base path
base_path = "/Users/nijat/Downloads/OneDrive_1_2024-09-21/Data"

# Load Clinical Data
clinical_data_path = os.path.join(base_path, "2019_TCGA-CDR-SupplementalTableS1.xlsx")
main_data_df = pd.read_excel(clinical_data_path)

# Load Tumor Classification Data
new_labels_path = os.path.join(base_path, "Matrix_WHO2021.csv")
new_categories = pd.read_csv(new_labels_path)

# Load Methylation Data (GBM & LGG)
gbm_data = pd.read_csv(os.path.join(base_path, "GBM_450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"))
lgg_data = pd.read_csv(os.path.join(base_path, "LGG-450K_Filtered_X-Y,SNPs,Non-overlap850K.csv"))

print("✅ Data Loaded Successfully!")

✅ Data Loaded Successfully!


## 🔍 Preprocess & Merge Data

In [4]:
# Rename column for consistency
main_data_df.rename(columns={'bcr_patient_barcode': 'Patient_ID'}, inplace=True)

# Filter Clinical Data for GBM & LGG tumor types
filtered_data_df = main_data_df[main_data_df['type'].isin(['GBM', 'LGG'])]

# Merge Clinical and Tumor Classification Data
merged_clinical_data = pd.merge(filtered_data_df, new_categories, on='Patient_ID', how='inner')

print(f"✅ Merged Clinical Data Shape: {merged_clinical_data.shape}")

✅ Merged Clinical Data Shape: (1110, 37)


In [5]:
# Function to preprocess methylation data (Transpose & Rename)
def preprocess_methylation_data(df):
    df_trns = df.T  # Transpose data (genes as columns)
    df_trns.columns = df_trns.iloc[0]  # First row as column names
    return df_trns[1:]  # Remove first row

# Process GBM & LGG Data
gbm_trns_final = preprocess_methylation_data(gbm_data)
lgg_trns_final = preprocess_methylation_data(lgg_data)

# Merge GBM & LGG Methylation Data
merged_methylation_data = pd.concat([gbm_trns_final, lgg_trns_final])

# Ensure "Patient_ID" is named correctly
merged_methylation_data.rename(columns={'Index': 'Patient_ID'}, inplace=True)

# Merge Methylation & Clinical Data
final_data = pd.merge(merged_methylation_data, merged_clinical_data, 
                      left_index=True, right_on='Patient_ID', 
                      how='inner')

print(f"✅ Final Merged Data Shape: {final_data.shape}")

✅ Final Merged Data Shape: (653, 403991)


In [9]:
final_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 653 entries, 368 to 1088
Columns: 403991 entries, cg13869341 to classification.2021_simplified.labels
dtypes: float64(14), int64(1), object(403976)
memory usage: 2.0+ GB


## 🧼 Clean Data & Encode Categorical Variables

In [10]:
columns_to_drop = [
    'DSS', 'DSS.time', 'PFI', 'PFI.time', 'DFI', 'DFI.time', 'Redaction',
    'TCGA-histological.type', 'classification.2021_complete.labels'
]

# Drop unnecessary columns
final_data_cleaned = final_data.drop(columns=columns_to_drop, errors="ignore")

# Print the new shape after column removal
print(f"📉 Cleaned Data Shape: {final_data_cleaned.shape}")

📉 Cleaned Data Shape: (653, 403982)


In [11]:
final_data_cleaned['classification.2021_simplified.labels'].value_counts()

classification.2021_simplified.labels
astrocytoma          256
glioblastoma         194
oligodendroglioma    169
unclassified          34
Name: count, dtype: int64